# SpamShield AI - DistilBERT Fine-Tuning

Full training pipeline: data processing, fine-tuning, evaluation, and export.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch transformers datasets scikit-learn pandas matplotlib seaborn

In [ ]:
import re
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertModel,
    DistilBertTokenizer,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'num_labels': 2,
    'max_length': 128,
    'dropout': 0.3,
    'batch_size': 32,
    'epochs': 5,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'early_stopping_patience': 2,
    'seed': 42,
    'save_dir': '/content/drive/My Drive/data/spamshield_model'
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))

## 2. Load Data

In [ ]:
DATA_PATH = '/content/drive/My Drive/data/smsspamcollection/SMSSpamCollection'

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Total: {len(df)}')
print(f'Ham: {(df["label"] == 0).sum()} ({(df["label"] == 0).mean()*100:.1f}%)')
print(f'Spam: {(df["label"] == 1).sum()} ({(df["label"] == 1).mean()*100:.1f}%)')
df.head()

## 3. Preprocess

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned'] = df['text'].apply(clean_text)
print('Sample cleaned texts:')
for i in range(3):
    print(f'  [{"SPAM" if df.iloc[i]["label"] == 1 else "HAM "}] {df.iloc[i]["cleaned"][:80]}')

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    df['cleaned'].tolist(), df['label'].tolist(),
    test_size=0.15, random_state=CONFIG['seed'], stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=CONFIG['seed'], stratify=y_temp
)

print(f'Train: {len(X_train)} ({sum(y_train)} spam)')
print(f'Val:   {len(X_val)} ({sum(y_val)} spam)')
print(f'Test:  {len(X_test)} ({sum(y_test)} spam)')

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG['model_name'])

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=CONFIG['max_length'], return_tensors=None)

train_enc = tokenize(X_train)
val_enc = tokenize(X_val)
test_enc = tokenize(X_test)

print(f'Tokenization complete. Max length: {CONFIG["max_length"]}')

## 4. Dataset & Model

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = SpamDataset(train_enc, y_train)
val_dataset = SpamDataset(val_enc, y_val)
test_dataset = SpamDataset(test_enc, y_test)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'])

print(f'DataLoaders ready. Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

In [ ]:
class SpamClassifier(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased', num_labels=2, dropout=0.3):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        self.dropout1 = nn.Dropout(dropout)
        self.fc1 = nn.Linear(self.distilbert.config.hidden_size, 256)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        x = self.dropout1(pooled)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout2(x)
        return self.fc2(x)

model = SpamClassifier(
    model_name=CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    dropout=CONFIG['dropout']
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable: {trainable_params:,}')

## 5. Training

In [ ]:
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
total_steps = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

print(f'Total steps: {total_steps}')
print(f'Warmup steps: {warmup_steps}')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'val_f1': []}
best_f1 = 0
patience_counter = 0

for epoch in range(CONFIG['epochs']):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            val_loss += criterion(logits, labels).item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Epoch {epoch+1}/{CONFIG["epochs"]} | '
          f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | '
          f'Acc: {val_acc:.4f} | F1: {val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  -> Saved (F1: {val_f1:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['early_stopping_patience']:
            print(f'Early stopping at epoch {epoch+1}')
            break

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
print(f'\nBest Val F1: {best_f1:.4f}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs, history['train_loss'], 'o-', label='Train', color='#3498db')
axes[0, 0].plot(epochs, history['val_loss'], 's-', label='Val', color='#e74c3c')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(epochs, history['val_accuracy'], 'o-', color='#2ecc71', lw=2)
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True)

axes[1, 0].plot(epochs, history['val_f1'], 's-', color='#e74c3c', lw=2)
axes[1, 0].set_title('Validation F1 Score')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('F1')
axes[1, 0].grid(True)

axes[1, 1].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#3498db')
axes[1, 1].plot(epochs, history['val_f1'], 's-', label='Val F1', color='#e74c3c')
axes[1, 1].set_title('Loss vs F1')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 7. Test Evaluation

In [ ]:
model.eval()
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print('Test Results:')
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1 Score:  {f1:.4f}')
print(f'  ROC-AUC:   {auc:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
            annot_kws={'size': 14})
axes[0].set_title('Confusion Matrix', fontsize=14)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_title('ROC Curve', fontsize=14)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
fp_idx = [i for i in range(len(y_test)) if y_true[i] == 0 and y_pred[i] == 1]
fn_idx = [i for i in range(len(y_test)) if y_true[i] == 1 and y_pred[i] == 0]

print(f'False Positives: {len(fp_idx)}')
for i in sorted(fp_idx, key=lambda x: y_prob[x], reverse=True)[:5]:
    print(f'  [{y_prob[i]:.3f}] {X_test[i][:80]}')

print(f'\nFalse Negatives: {len(fn_idx)}')
for i in sorted(fn_idx, key=lambda x: y_prob[x])[:5]:
    print(f'  [{y_prob[i]:.3f}] {X_test[i][:80]}')

In [ ]:
correct_mask = [t == p for t, p in zip(y_true, y_pred)]
correct_conf = [max(y_prob[i], 1 - y_prob[i]) for i in range(len(y_prob)) if correct_mask[i]]
incorrect_conf = [max(y_prob[i], 1 - y_prob[i]) for i in range(len(y_prob)) if not correct_mask[i]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist([y_prob[i] for i in range(len(y_prob)) if y_true[i] == 1], bins=30, alpha=0.6, color='#e74c3c', label='Spam', density=True)
axes[0].hist([y_prob[i] for i in range(len(y_prob)) if y_true[i] == 0], bins=30, alpha=0.6, color='#2ecc71', label='Ham', density=True)
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Threshold')
axes[0].set_title('Score Distribution by Class')
axes[0].set_xlabel('P(Spam)')
axes[0].legend()

axes[1].hist(correct_conf, bins=30, alpha=0.7, color='#2ecc71', label='Correct')
if incorrect_conf:
    axes[1].hist(incorrect_conf, bins=30, alpha=0.7, color='#e74c3c', label='Incorrect')
axes[1].set_title('Confidence Distribution')
axes[1].set_xlabel('Confidence')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Manual Predictions

In [ ]:
def predict(text):
    enc = tokenizer(text, truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors='pt')
    input_ids = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    return {'label': 'Spam' if pred == 1 else 'Ham', 'confidence': probs[0][pred].item()}

tests = [
    'Hey, are we still meeting for lunch tomorrow?',
    "Congratulations! You've won a $1000 gift card. Click here to claim now!",
    'URGENT: Your account has been compromised. Verify your password immediately.',
    'Can you pick up some milk on your way home?',
    'FREE FREE FREE! Win a car by texting WIN to 80085',
    'Meeting moved to 3pm. See you there.',
    'You have been selected for a cash prize! Call now!',
    'Thanks for dinner last night, it was great!',
    'WINNER!! As a valued network customer you have been selected to receive a £900 prize reward!',
    'Hi mom, I\'ll be home for dinner around 7pm.'
]

print(f'{"Label":<6} {"Conf":<8} Message')
print('-' * 80)
for msg in tests:
    r = predict(msg)
    print(f'{r["label"]:<6} {r["confidence"]:.2%}   {msg[:60]}')

## 10. Save Model

In [ ]:
import tarfile

save_dir = CONFIG['save_dir']
!mkdir -p $save_dir

torch.save(model.state_dict(), f'{save_dir}/model.pt')
tokenizer.save_pretrained(save_dir)

results = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'roc_auc': auc,
    'history': history,
    'config': CONFIG
}
with open(f'{save_dir}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {save_dir}')
print(f'Files: model.pt, tokenizer files, results.json')

## 11. Package for SageMaker

In [ ]:
import tarfile
import os

tar_path = f'{save_dir}/model.tar.gz'

with tarfile.open(tar_path, 'w:gz') as tar:
    for root, dirs, files in os.walk(save_dir):
        for file in files:
            if file == 'model.tar.gz':
                continue
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, save_dir)
            tar.add(file_path, arcname=arcname)

tar_size = os.path.getsize(tar_path) / (1024 * 1024)
print(f'Created: {tar_path}')
print(f'Size: {tar_size:.2f} MB')
print(f'\nContents:')
with tarfile.open(tar_path, 'r') as tar:
    for member in tar.getmembers():
        print(f'  {member.name} ({member.size / 1024:.1f} KB)')

In [ ]:
# Upload to S3 (uncomment and set your bucket)
# import boto3
#
# s3 = boto3.client('s3')
# BUCKET = 'spamshield-ai-models'
# S3_KEY = 'model/model.tar.gz'
#
# s3.upload_file(tar_path, BUCKET, S3_KEY)
# print(f'Uploaded to s3://{BUCKET}/{S3_KEY}')